# NanoServe — Week 1: Fused Softmax & LayerNorm in Triton

**Goal for the week (v0.1):** write fused GPU kernels for softmax and LayerNorm in Triton, prove they are correct, and measure how close they get to the GPU's peak memory bandwidth.

**Before you start:** in Colab, go to *Runtime → Change runtime type → T4 GPU*.

| Day | Section in this notebook | Read / watch first |
|---|---|---|
| 1 | Setup + vector add | "Making Deep Learning Go Brrrr" · Triton tutorial 01 |
| 2 | Benchmark harness | Triton-Puzzles (first few) |
| 3 | Fused softmax | Triton tutorial 02 |
| 4 | Online softmax | Online softmax paper, sections 1–3 |
| 5 | LayerNorm | Triton tutorial 05 (forward only) |
| 6 | Profiling | PyTorch profiler recipe |
| 7 | README | Template at the bottom |

Rule for the week: **correct first, fast second.** Every kernel must pass its correctness check before you benchmark it.

---
## Day 1 — Setup and your first kernel

Read first: [Making Deep Learning Go Brrrr From First Principles](https://horace.io/brrr_intro.html) — focus on the *Bandwidth* section.
Then: [Triton tutorial 01 — Vector Addition](https://triton-lang.org/main/getting-started/tutorials/01-vector-add.html).

In [4]:
!nvidia-smi

import torch, triton, triton.language as tl
import pandas as pd
import matplotlib.pyplot as plt

assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU"
DEVICE = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)

# Theoretical peak memory bandwidth (GB/s) from NVIDIA spec sheets.
PEAK_GBPS = {"Tesla T4": 320, "NVIDIA L4": 300, "NVIDIA A100-SXM4-40GB": 1555}
PEAK = PEAK_GBPS.get(GPU_NAME)

print(f"GPU: {GPU_NAME} | peak bandwidth: {PEAK} GB/s")
print(f"torch {torch.__version__} | triton {triton.__version__}")

/bin/bash: line 1: nvidia-smi: command not found


ModuleNotFoundError: No module named 'triton'

In [ ]:
import torch , triton, triton.language as tl
import pandas as pd
import matplotlib.pyplot as plt


### Vector add — the "hello world" of GPU kernels

Each **program** (a block of GPU threads) handles `BLOCK_SIZE` elements. `tl.program_id(0)` tells a program which chunk is its own. The **mask** stops the last program from reading past the end of the array.

Type this out yourself rather than copy-pasting — it's the pattern every kernel this week uses.

In [ ]:
@triton.jit
def add_kernel(x_ptr, y_ptr, out_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(axis=0)                     # which chunk am I?
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements                     # guard the tail
    x = tl.load(x_ptr + offsets, mask=mask)
    y = tl.load(y_ptr + offsets, mask=mask)
    tl.store(out_ptr + offsets, x + y, mask=mask)


def triton_add(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    out = torch.empty_like(x)
    n = out.numel()
    grid = lambda meta: (triton.cdiv(n, meta["BLOCK_SIZE"]),)   # number of programs
    add_kernel[grid](x, y, out, n, BLOCK_SIZE=1024)
    return out


x = torch.randn(98_432, device=DEVICE)   # deliberately not a multiple of 1024
y = torch.randn(98_432, device=DEVICE)
torch.testing.assert_close(triton_add(x, y), x + y)
print("vector add: correct")

---
## Day 2 — A benchmark harness you'll reuse all week

Three rules of GPU timing:
1. GPU calls are **asynchronous** — Python returns before the GPU finishes. `do_bench` synchronises for you.
2. **Warm up** first — the first call includes compilation.
3. Report the **median** and a spread, not one number.

We report **effective bandwidth** = ideal bytes moved ÷ time. "Ideal bytes" means reading the input once and writing the output once. A kernel that wastes trips to memory shows up as low GB/s.

Practice: try the first few [Triton-Puzzles](https://github.com/srush/Triton-Puzzles) to get fluent with `tl.load`, masks and `tl.arange`.

In [ ]:
def gbps(nbytes: int, ms: float) -> float:
    return nbytes / (ms * 1e-3) / 1e9


def sweep(impls: dict, make_input, sizes, nbytes_fn, title: str, xlabel: str) -> pd.DataFrame:
    # impls: {"name": fn}. make_input(size) -> tuple of args. nbytes_fn(*args) -> ideal bytes.
    rows = []
    for size in sizes:
        args = make_input(size)
        nbytes = nbytes_fn(*args)
        for name, fn in impls.items():
            med, lo, hi = triton.testing.do_bench(lambda: fn(*args), quantiles=[0.5, 0.2, 0.8])
            rows.append({"size": size, "impl": name, "ms": med, "GB/s": gbps(nbytes, med)})
    df = pd.DataFrame(rows)

    fig, ax = plt.subplots(figsize=(7, 4))
    for name, g in df.groupby("impl"):
        ax.plot(g["size"], g["GB/s"], marker="o", label=name)
    if PEAK:
        ax.axhline(PEAK, linestyle="--", color="gray", label=f"peak ({PEAK} GB/s)")
    ax.set_xscale("log", base=2)
    ax.set_xlabel(xlabel); ax.set_ylabel("effective bandwidth (GB/s)")
    ax.set_title(title); ax.legend(); ax.grid(alpha=0.3)
    plt.show()
    return df.pivot(index="size", columns="impl", values="GB/s").round(1)


add_results = sweep(
    impls={"torch": lambda a, b: a + b, "triton": triton_add},
    make_input=lambda n: (torch.randn(n, device=DEVICE), torch.randn(n, device=DEVICE)),
    sizes=[2**i for i in range(12, 28, 2)],
    nbytes_fn=lambda a, b: 3 * a.numel() * a.element_size(),   # read a, read b, write out
    title="Vector add", xlabel="number of elements",
)
add_results

**What to notice:** at small sizes bandwidth is low for *both* versions — the GPU spends its time launching the kernel, not moving data. At large sizes both should approach the peak line. Write down the best GB/s you hit; it's your realistic ceiling on this GPU.

---
## Day 3 — Fused softmax

Read first: [Triton tutorial 02 — Fused Softmax](https://triton-lang.org/main/getting-started/tutorials/02-fused-softmax.html). Try to fill in the TODOs **before** looking at the tutorial's kernel — use it as an answer key when stuck.

Plan: one program per row. Load the row once, do everything in fast on-chip memory, write once.

In [ ]:
def naive_softmax(x: torch.Tensor) -> torch.Tensor:
    # Each line is a separate GPU kernel that reads from and writes to slow global memory.
    x_max = x.max(dim=1)[0]
    z = x - x_max[:, None]
    numerator = torch.exp(z)
    denominator = numerator.sum(dim=1)
    return numerator / denominator[:, None]

In [ ]:
@triton.jit
def softmax_kernel(out_ptr, in_ptr, in_row_stride, out_row_stride, n_cols,
                   BLOCK_SIZE: tl.constexpr):
    row = tl.program_id(0)                       # one program per row
    cols = tl.arange(0, BLOCK_SIZE)
    mask = cols < n_cols

    # TODO 1: load the row from in_ptr + row * in_row_stride + cols.
    #         Padding lanes must not win the max or add to the sum:
    #         use mask=mask, other=-float("inf")

    row_start_ptr = in_ptr + row * in_row_stride
    x = tl.load(row_start_ptr+cols, mask = mask, other = -float("inf"))


    # TODO 2: subtract the row max (tl.max(x, axis=0)) for numerical stability
    row_max = tl.max(x, axis=0)
    z = x - row_max

    # TODO 3: exponentiate (tl.exp), sum (tl.sum), divide
    numerator = tl.exp(z)
    denominator = tl.sum(numerator, axis=0)
    y = numerator / denominator

    # TODO 4: store y to out_ptr + row * out_row_stride + cols, with mask=mask
    tl.store(out_ptr + row * out_row_stride + cols,y,mask = mask)



def triton_softmax(x: torch.Tensor) -> torch.Tensor:
    n_rows, n_cols = x.shape
    BLOCK_SIZE = max(triton.next_power_of_2(n_cols), 16)   # whole row must fit in one block
    num_warps = 4 if BLOCK_SIZE <= 2048 else (8 if BLOCK_SIZE <= 4096 else 16)
    y = torch.empty_like(x)
    softmax_kernel[(n_rows,)](y, x, x.stride(0), y.stride(0), n_cols,
                              BLOCK_SIZE=BLOCK_SIZE, num_warps=num_warps)
    return y

In [ ]:
# Correctness — awkward sizes on purpose
for shape in [(1, 1), (7, 13), (1823, 781), (4096, 4096)]:
    x = torch.randn(shape, device=DEVICE)
    torch.testing.assert_close(triton_softmax(x), torch.softmax(x, dim=1))
print("softmax: correct")

softmax_results = sweep(
    impls={"naive (torch ops)": naive_softmax,
           "torch.softmax": lambda x: torch.softmax(x, dim=1),
           "triton fused": triton_softmax},
    make_input=lambda n: (torch.randn(4096, n, device=DEVICE),),
    sizes=[2**i for i in range(7, 14)],          # 128 .. 8192 columns
    nbytes_fn=lambda x: 2 * x.numel() * x.element_size(),   # read once, write once
    title="Softmax, 4096 rows", xlabel="number of columns",
)
softmax_results

**Questions to answer in your README:**
- How many times faster is the fused kernel than the naive one? Count how many times the naive version reads and writes the full matrix — does the speedup roughly match?
- Where does your kernel beat `torch.softmax`, and where does it lose?
- What happens as the number of columns grows? (Hint: the whole row has to fit in on-chip memory. Tomorrow fixes this.)

---
## Day 4 — Online softmax

Read first: [Online normalizer calculation for softmax (Milakov & Gimelshein, 2018)](https://arxiv.org/abs/1805.02867), sections 1–3.

Yesterday's kernel needs the whole row in one block, so it breaks down for very long rows. **Online softmax** walks the row in chunks, keeping a running max `m` and running sum `d`. When a new chunk raises the max, the old sum is rescaled:

$$m_{new} = \max(m, \max(x_{chunk})) \qquad d_{new} = d \cdot e^{\,m - m_{new}} + \sum e^{\,x_{chunk} - m_{new}}$$

This exact trick is the heart of FlashAttention — get it solid now.

Step 1 is a plain PyTorch version so you can see the maths work before writing any kernel.

In [ ]:
def online_softmax_reference(x: torch.Tensor, block: int = 256) -> torch.Tensor:
    # Row-by-row, chunk-by-chunk. Slow, but it mirrors the kernel exactly.
    out = torch.empty_like(x)
    for r in range(x.shape[0]):
        row = x[r]
        # first chunk initialises m and d
        chunk = row[:block]
        m = chunk.max()
        d = torch.exp(chunk - m).sum()
        # remaining chunks update them
        for start in range(block, row.numel(), block):
            chunk = row[start:start + block]
            m_new = torch.maximum(m, chunk.max())
            d = d * torch.exp(m - m_new) + torch.exp(chunk - m_new).sum()
            m = m_new
        out[r] = torch.exp(row - m) / d   # second pass: normalise
    return out


x = torch.randn(8, 1000, device=DEVICE)
torch.testing.assert_close(online_softmax_reference(x), torch.softmax(x, dim=1))
print("online softmax maths: correct")

In [ ]:
@triton.jit
def online_softmax_kernel(out_ptr, in_ptr, in_row_stride, out_row_stride, n_cols,
                          BLOCK_SIZE: tl.constexpr):
    row = tl.program_id(0)
    row_in = in_ptr + row * in_row_stride
    row_out = out_ptr + row * out_row_stride
    cols = tl.arange(0, BLOCK_SIZE)

    # Pass 1a: the first chunk initialises m and d (same as the reference above).
    mask = cols < n_cols
    x = tl.load(row_in + cols, mask=mask, other=-float("inf"))
    m = tl.max(x, axis=0)
    d = tl.sum(tl.exp(x - m), axis=0)

    # Pass 1b: remaining chunks update m and d.
    for start in range(BLOCK_SIZE, n_cols, BLOCK_SIZE):
        offs = start + cols

        # TODO 1: load the chunk (mask=offs < n_cols, other=-inf)
        chunk = tl.load(row_in + offs, mask=offs<n_cols, other=-float("inf"))
        # TODO 2: m_new = tl.maximum(m, tl.max(chunk, axis=0))
        m_new = tl.maximum(m, tl.max(chunk, axis=0))
        # TODO 3: d = d * tl.exp(m - m_new) + tl.sum(tl.exp(chunk - m_new), axis=0)
        d = d * tl.exp(m - m_new) + tl.sum(tl.exp(chunk - m_new), axis=0)
        # TODO 4: m = m_new
        m = m_new

    # Pass 2: write the normalised output, chunk by chunk.
    for start in range(0, n_cols, BLOCK_SIZE):
        offs = start + cols
        mask = offs < n_cols
        # TODO 5: load the chunk, compute tl.exp(chunk - m) / d, store with the mask
        chunk = tl.load(row_in + offs, mask=mask, other=-float("inf"))
        y = tl.exp(chunk - m) / d
        tl.store(row_out + offs, y, mask=mask)


def triton_online_softmax(x: torch.Tensor, BLOCK_SIZE: int = 1024) -> torch.Tensor:
    n_rows, n_cols = x.shape
    y = torch.empty_like(x)
    online_softmax_kernel[(n_rows,)](y, x, x.stride(0), y.stride(0), n_cols,
                                     BLOCK_SIZE=BLOCK_SIZE, num_warps=4)
    return y

In [ ]:
for shape in [(3, 5), (17, 1025), (512, 20_000)]:
    x = torch.randn(shape, device=DEVICE)
    # slightly looser tolerance: summing in a different order changes the last digits
    torch.testing.assert_close(triton_online_softmax(x), torch.softmax(x, dim=1), rtol=1e-4, atol=1e-5)
print("online softmax kernel: correct")

online_results = sweep(
    impls={"torch.softmax": lambda x: torch.softmax(x, dim=1),
           "triton fused (day 3)": triton_softmax,
           "triton online": triton_online_softmax},
    make_input=lambda n: (torch.randn(1024, n, device=DEVICE),),
    sizes=[2**i for i in range(10, 17)],         # up to 65536 columns
    nbytes_fn=lambda x: 2 * x.numel() * x.element_size(),
    title="Softmax on long rows, 1024 rows", xlabel="number of columns",
)
online_results

**Questions for the README:** the online kernel reads the input twice (pass 1 and pass 2), so its *ideal* is not quite the same as the day-3 kernel's. At which row length does it start beating the day-3 kernel, and why? Try `BLOCK_SIZE` of 512, 1024 and 2048 — which is best?

---
## Day 5 — Fused LayerNorm (forward)

Read first: [Triton tutorial 05 — Layer Normalization](https://triton-lang.org/main/getting-started/tutorials/05-layer-norm.html), forward pass only. Skip the backward pass this week.

$$y = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} \cdot w + b$$

Same shape as softmax: one program per row, two reductions (mean, variance), then an elementwise step.

In [ ]:
@triton.jit
def layernorm_kernel(out_ptr, in_ptr, w_ptr, b_ptr, row_stride, n_cols, eps,
                     BLOCK_SIZE: tl.constexpr):
    row = tl.program_id(0)
    cols = tl.arange(0, BLOCK_SIZE)
    mask = cols < n_cols
    x = tl.load(in_ptr + row * row_stride + cols, mask=mask, other=0.0).to(tl.float32)

    # TODO 1: mean = tl.sum(x, axis=0) / n_cols
    mean = tl.sum(x, axis=0)/n_cols
    # TODO 2: variance. Careful: padding lanes hold 0, so (x - mean) is NOT zero there.
    #         Use diff = tl.where(mask, x - mean, 0.0), then var = tl.sum(diff * diff, axis=0) / n_cols'
    diff = tl.where(mask, x-mean,0.0)
    var = tl.sum(diff*diff, axis=0)/n_cols
    # TODO 3: rstd = tl.rsqrt(var + eps)
    rstd = tl.rsqrt(var+eps)
    # TODO 4: load w and b (mask=mask), compute y = diff * rstd * w + b
    w = tl.load(w_ptr+cols, mask=mask, other = 0.0)
    b = tl.load(b_ptr+cols, mask=mask, other = 0.0)

    # TODO 5: store y to out_ptr + row * row_stride + cols, with the mask
    y = diff* rstd * w+b
    tl.store(out_ptr + row * row_stride + cols,y, mask = mask)


def triton_layernorm(x, w, b, eps=1e-5):
    n_rows, n_cols = x.shape
    BLOCK_SIZE = max(triton.next_power_of_2(n_cols), 16)
    num_warps = 4 if BLOCK_SIZE <= 2048 else (8 if BLOCK_SIZE <= 4096 else 16)
    y = torch.empty_like(x)
    layernorm_kernel[(n_rows,)](y, x, w, b, x.stride(0), n_cols, eps,
                                BLOCK_SIZE=BLOCK_SIZE, num_warps=num_warps)
    return y

In [ ]:
import torch.nn.functional as F

def make_ln_input(n, rows=4096):
    x = torch.randn(rows, n, device=DEVICE)
    w = torch.rand(n, device=DEVICE)
    b = torch.randn(n, device=DEVICE)
    return x, w, b

for n in [1, 13, 781, 4096]:
    x, w, b = make_ln_input(n, rows=37)
    torch.testing.assert_close(triton_layernorm(x, w, b), F.layer_norm(x, (n,), w, b, eps=1e-5),
                               rtol=1e-4, atol=1e-4)   # different summation order
print("layernorm: correct")

ln_results = sweep(
    impls={"torch layer_norm": lambda x, w, b: F.layer_norm(x, (x.shape[1],), w, b, eps=1e-5),
           "triton fused": triton_layernorm},
    make_input=make_ln_input,
    sizes=[2**i for i in range(7, 14)],
    nbytes_fn=lambda x, w, b: 2 * x.numel() * x.element_size(),   # w, b are tiny
    title="LayerNorm forward, 4096 rows", xlabel="number of columns",
)
ln_results

---
## Day 6 — Profiling

Read first: [PyTorch profiler recipe](https://pytorch.org/tutorials/recipes/recipes/profiler_recipe.html).

The profiler shows every GPU kernel that actually ran. Count the kernels in the naive softmax versus your fused one — that count is your memory-traffic argument, now backed by evidence.

In [ ]:
from torch.profiler import profile, ProfilerActivity

x = torch.randn(4096, 4096, device=DEVICE)
for _ in range(3):                      # warm-up so compilation isn't profiled
    naive_softmax(x); triton_softmax(x)
torch.cuda.synchronize()

with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
    for _ in range(10):
        naive_softmax(x)
        triton_softmax(x)
    torch.cuda.synchronize()

events = prof.key_averages()
try:
    print(events.table(sort_by="device_time_total", row_limit=15))
except Exception:                       # older torch versions use the old name
    print(events.table(sort_by="cuda_time_total", row_limit=15))

prof.export_chrome_trace("softmax_trace.json")
print("Download softmax_trace.json and open it at https://ui.perfetto.dev to see the timeline.")

**Stretch (optional):** run Nsight Compute on one kernel to see achieved bandwidth and occupancy directly. It can be fiddly on Colab — skip it if it costs more than an hour this week.

---
## Day 7 — Write the README

Copy this into `README.md` in your GitHub repo and fill it in. Your plots go in an `assets/` folder.

```markdown
# NanoServe

A minimal LLM inference engine built from scratch on custom Triton kernels.

## v0.1 — fused kernels
| Kernel | Best effective bandwidth | % of T4 peak (320 GB/s) | vs naive PyTorch |
|---|---|---|---|
| Fused softmax | ... GB/s | ...% | ...x faster |
| Online softmax | ... GB/s | ...% | ... |
| LayerNorm (fwd) | ... GB/s | ...% | ... |

![softmax benchmark](assets/softmax.png)

## Why fusion helps
<!-- 1 paragraph: memory-bound ops, how many HBM round trips naive softmax makes,
     the profiler kernel count as evidence -->

## Online softmax
<!-- the running max / rescaled sum idea in 3-4 lines, and when it wins -->

## What I learned / what surprised me

## Next: v0.2 — FlashAttention forward kernel
```